# Data Generation

This notebook is the executable entry point for raw SLIDE simulations. It generates the raw products consumed by `data_processing.ipynb` and writes them into `raw_data/` with parameter-descriptive filenames. Reusable simulation kernels live in `slide.data_generation`.

## Setup

Import reusable kernels, create output directories, and show where generated files will be written.

In [ ]:
import jax as jr

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import jax.random as jr
import numpy as np
from tqdm.auto import tqdm

from slide.data_generation import (
    EMPIRICAL_NAMES,
    GENERATION_STEPS,
    RAW_FILENAMES,
    all_start_locs,
    expected_raw_outputs,
    generate_empirical_decay_curves,
    generate_empirical_strategy_sweep,
    generate_nk_decay_curves,
    generate_nk_strategy_sweep,
    load_empirical_landscape,
    missing_raw_outputs,
    nk_grid_pairs,
    run_nk_diffusion_replicates,
    strategy_grid,
    uniform_start_locs,
)
from slide.utils import get_figures_dir, get_processed_data_dir, get_raw_data_dir, save_raw

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


raw_data: /home/lady5906/workspace_python/SLIDE/raw_data
processed_data: /home/lady5906/workspace_python/SLIDE/processed_data
figures: /home/lady5906/workspace_python/SLIDE/figures


## Expected Raw Products

These are the raw products loaded by `data_processing.ipynb` and required for the paper figures.

In [ ]:
print(f"Expected raw products: {len(RAW_FILENAMES)}")
for key in sorted(RAW_FILENAMES):
    print(f"{key}: {RAW_FILENAMES[key]}")

## NK Grid Decay - Figure 3

Generate the large NK no-selection diffusion grid used for ruggedness accuracy, example decay curves, and NK summaries in Figure 3.

- Paper reference: Figure 3A-B/E.
- Product: `nk_decay_grid`, saved to `RAW_FILENAMES["nk_decay_grid"]`.
- Core variables: NK landscape size `N`, epistasis `K`, alleles `A`, total mutation rate `mutation_rate`, population size `popsize`, and generations `M`.
- Each diffusion starts from the all-zero genotype and records the mean population fitness trajectory $F_\mu$.


In [ ]:
# NK landscape grid parameters
N_range = (10, 50)
num_grid_samples = 10
A = 2
nk_pairs = nk_grid_pairs(N_range, num_grid_samples)

# Diffusion experiment parameters
mutation_rate = 0.5
popsize = 2500
num_landscapes = 25
num_reps_per_landscape = 10
M = 25
seed = 42
start_policy = "zero"

rep_keys = jr.split(jr.PRNGKey(seed), num_landscapes)
nk_decay_grid = []

for N, K in tqdm(nk_pairs, desc="NK decay grid"):
    pair_results = []
    for key in rep_keys:
        run = run_nk_diffusion_replicates(
            key,
            n_sites=N,
            k=K,
            num_alleles=A,
            start=np.zeros(N, dtype=np.int32),
            popsize=popsize,
            mutation_rate=mutation_rate / N,
            num_reps=num_reps_per_landscape,
            num_steps=M,
        )
        pair_results.append(run["fitness"].mean(axis=-1))
    nk_decay_grid.append(np.asarray(pair_results))

nk_decay_grid_payload = {
    "data": np.asarray(nk_decay_grid),
    "params": {
        "N_range": N_range,
        "num_grid_samples": num_grid_samples,
        "nk_pairs": nk_pairs,
        "A": A,
        "mutation_rate": mutation_rate,
        "popsize": popsize,
        "num_landscapes": num_landscapes,
        "num_reps_per_landscape": num_reps_per_landscape,
        "M": M,
        "seed": seed,
        "start_policy": start_policy,
    },
    "metadata": {
        "description": "NK no-selection diffusion grid for ruggedness accuracy and example decay curves.",
        "paper_reference": "Figure 3A-B/E",
        "output_key": "nk_decay_grid",
        "filename": RAW_FILENAMES["nk_decay_grid"],
    },
}
save_raw(nk_decay_grid_payload, RAW_FILENAMES["nk_decay_grid"])


## NK Strategy Grid - Figure 5A

Generate the NK strategy lookup grid used to connect fitted ruggedness to directed-evolution control parameters.

- Paper reference: Figure 5A.
- Product: `nk_strategy_grid`, saved to `RAW_FILENAMES["nk_strategy_grid"]`.
- Core variables: NK grid `N, K, A`, strategy grid size, total mutation rate `mutation_rate`, total population size `popsize`, and generations `M`.
- The saved payload records the split and base-chance grid used for downstream strategy labels.


In [ ]:
# NK landscape grid parameters
N_range = (10, 50)
num_grid_samples = 10
A = 2
nk_pairs = nk_grid_pairs(N_range, num_grid_samples)

# Directed-evolution strategy sweep parameters
mutation_rate = 0.1
popsize = 1200
num_landscapes_per_pair = 1
num_reps = 25
M = 25
strategy_grid_size = 7
outer_reps = 1
seed = 42
thresholds, base_chances, splits = strategy_grid(strategy_grid_size)

nk_strategy_grid = []
for N, K in tqdm(nk_pairs, desc="NK strategy grid"):
    pair = generate_nk_strategy_sweep(
        n_sites=N,
        num_alleles=A,
        k_values=[K],
        mutation_rate=mutation_rate,
        popsize=popsize,
        num_landscapes=num_landscapes_per_pair,
        num_reps=num_reps,
        num_steps=M,
        strategy_grid_size=strategy_grid_size,
        outer_reps=outer_reps,
        seed=seed,
    )
    nk_strategy_grid.append(pair.reshape(strategy_grid_size, strategy_grid_size, num_reps))

nk_strategy_grid_payload = {
    "data": np.asarray(nk_strategy_grid),
    "params": {
        "N_range": N_range,
        "num_grid_samples": num_grid_samples,
        "nk_pairs": nk_pairs,
        "A": A,
        "mutation_rate": mutation_rate,
        "popsize": popsize,
        "num_landscapes_per_pair": num_landscapes_per_pair,
        "num_reps": num_reps,
        "M": M,
        "strategy_grid_size": strategy_grid_size,
        "outer_reps": outer_reps,
        "thresholds": np.asarray(thresholds),
        "base_chances": np.asarray(base_chances),
        "splits": splits,
        "seed": seed,
    },
    "metadata": {
        "description": "NK directed-evolution strategy lookup grid.",
        "paper_reference": "Figure 5A",
        "output_key": "nk_strategy_grid",
        "filename": RAW_FILENAMES["nk_strategy_grid"],
    },
}
save_raw(nk_strategy_grid_payload, RAW_FILENAMES["nk_strategy_grid"])


## NK Accuracy Sweeps - Figure 3C-D

Generate population-size and mutation-rate sensitivity sweeps for the fitted ruggedness estimate.

- Paper reference: Figure 3C-D.
- Products: `nk_popsize_accuracy` and `nk_mutation_accuracy`, saved to their `RAW_FILENAMES` entries.
- The NK landscape parameters are fixed as `N`, `K`, and `A`, and `num_lscapes` landscapes with these parameters are tested.
- Diffusion experiments are repeated `num_reps_per_lscape` times per landscape. The resulting trajectories are averaged to obtain $F_\mu$.
- A diffusion experiment starts with a single-genotype population of size `popsize` and applies unbiased mutations with mutation rate `mutation_rate` for `M` generations.


In [ ]:
# NK landscape parameters
N, K, A = 25, 15, 2

# Diffusion experiment parameters
num_lscapes, num_reps_per_lscape, M = 25, 20, 25
seed = 42
fit_mutation_scale = 1.0

# Use either [0,...,0] or random starting genotypes for the diffusion experiments
random_starting_genotype = True
start_policy = "random" if random_starting_genotype else "zero"

# Landscape generation keys
rep_keys = jr.split(jr.PRNGKey(seed), num_lscapes)

# Population sweep
pop_sizes = np.logspace(start=1, stop=6, num=6, endpoint=True, dtype=int)
popsize_mutation_rate = 0.5
popsize_accuracy = []
for popsize in tqdm(pop_sizes, desc="NK popsize accuracy"):
    pop_results = []
    for key in rep_keys:
        start = (
            np.asarray(jr.randint(key, (N,), 0, A), dtype=np.int32)
            if random_starting_genotype
            else np.zeros(N, dtype=np.int32)
        )
        run = run_nk_diffusion_replicates(
            key,
            n_sites=N,
            k=K,
            num_alleles=A,
            start=start,
            popsize=int(popsize),
            mutation_rate=popsize_mutation_rate / N,
            num_reps=num_reps_per_lscape,
            num_steps=M,
        )
        pop_results.append(run["fitness"].mean(axis=-1))
    popsize_accuracy.append(np.asarray(pop_results))

popsize_accuracy_payload = {
    "data": np.asarray(popsize_accuracy),
    "params": {
        "N": N,
        "K": K,
        "A": A,
        "num_lscapes": num_lscapes,
        "num_reps_per_lscape": num_reps_per_lscape,
        "M": M,
        "seed": seed,
        "start_policy": start_policy,
        "random_starting_genotype": random_starting_genotype,
        "pop_sizes": pop_sizes,
        "mutation_rate": popsize_mutation_rate,
        "fit_mutation_scale": fit_mutation_scale,
    },
    "metadata": {
        "description": "NK population-size sensitivity sweep for fitted ruggedness.",
        "paper_reference": "Figure 3C",
        "output_key": "nk_popsize_accuracy",
        "filename": RAW_FILENAMES["nk_popsize_accuracy"],
    },
}
save_raw(popsize_accuracy_payload, RAW_FILENAMES["nk_popsize_accuracy"])

# Mutation-rate sweep
popsize = 2000
mutation_rates = np.linspace(0.01, 2, 25)
mutation_accuracy = []
for mutation_rate in tqdm(mutation_rates, desc="NK mutation-rate accuracy"):
    mu_results = []
    for key in rep_keys:
        start = (
            np.asarray(jr.randint(key, (N,), 0, A), dtype=np.int32)
            if random_starting_genotype
            else np.zeros(N, dtype=np.int32)
        )
        run = run_nk_diffusion_replicates(
            key,
            n_sites=N,
            k=K,
            num_alleles=A,
            start=start,
            popsize=popsize,
            mutation_rate=float(mutation_rate) / N,
            num_reps=num_reps_per_lscape,
            num_steps=M,
        )
        mu_results.append(run["fitness"].mean(axis=-1))
    mutation_accuracy.append(np.asarray(mu_results))

mutation_accuracy_payload = {
    "data": np.asarray(mutation_accuracy),
    "params": {
        "N": N,
        "K": K,
        "A": A,
        "num_lscapes": num_lscapes,
        "num_reps_per_lscape": num_reps_per_lscape,
        "M": M,
        "seed": seed,
        "start_policy": start_policy,
        "random_starting_genotype": random_starting_genotype,
        "popsize": popsize,
        "mutation_rates": mutation_rates,
    },
    "metadata": {
        "description": "NK mutation-rate sensitivity sweep for fitted ruggedness.",
        "paper_reference": "Figure 3D",
        "output_key": "nk_mutation_accuracy",
        "filename": RAW_FILENAMES["nk_mutation_accuracy"],
    },
}
save_raw(mutation_accuracy_payload, RAW_FILENAMES["nk_mutation_accuracy"])


## N4A20 Decay And Strategy Sweeps - Figure 5

Generate the `N=4`, `A=20` NK decay and strategy products used for optimal-strategy summaries and the generation-count comparison.

- Paper reference: Figure 5A and generation-count support analyses.
- Products: `nk_decay_N4_A20`, `nk_strategy_N4_A20`, and `nk_strategy_N4_A20_steps{M}`.
- Core variables: `N`, `A`, `K_values`, total mutation rate `mutation_rate`, population size `popsize`, number of starts/landscapes, strategy grid size, and generations `M`.


In [ ]:
# Shared N4A20 parameters
N, A = 4, 20
K_values = [1, 2, 3]
mutation_rate = 0.1
popsize = 1200
seed = 42

# Decay parameters
num_starts = 10000
num_decay_reps = 10
M = 25
nk_decay_N4_A20 = generate_nk_decay_curves(
    n_sites=N,
    num_alleles=A,
    k_values=K_values,
    mutation_rate=mutation_rate,
    popsize=popsize,
    num_starts=num_starts,
    num_reps=num_decay_reps,
    num_steps=M,
    seed=seed,
)
nk_decay_N4_A20_payload = {
    "data": nk_decay_N4_A20,
    "params": {
        "N": N,
        "A": A,
        "K_values": K_values,
        "mutation_rate": mutation_rate,
        "popsize": popsize,
        "num_starts": num_starts,
        "num_reps": num_decay_reps,
        "M": M,
        "seed": seed,
    },
    "metadata": {
        "description": "N4A20 NK no-selection decay curves for optimal-strategy summaries.",
        "paper_reference": "Figure 5",
        "output_key": "nk_decay_N4_A20",
        "filename": RAW_FILENAMES["nk_decay_N4_A20"],
    },
}
save_raw(nk_decay_N4_A20_payload, RAW_FILENAMES["nk_decay_N4_A20"])

# Strategy sweep parameters
num_strategy_landscapes = 125
num_strategy_reps = 10
strategy_grid_size = 7
outer_reps = 10
thresholds, base_chances, splits = strategy_grid(strategy_grid_size)
nk_strategy_N4_A20 = generate_nk_strategy_sweep(
    n_sites=N,
    num_alleles=A,
    k_values=K_values,
    mutation_rate=mutation_rate,
    popsize=popsize,
    num_landscapes=num_strategy_landscapes,
    num_reps=num_strategy_reps,
    num_steps=M,
    strategy_grid_size=strategy_grid_size,
    outer_reps=outer_reps,
    seed=seed,
)
nk_strategy_N4_A20_payload = {
    "data": nk_strategy_N4_A20,
    "params": {
        "N": N,
        "A": A,
        "K_values": K_values,
        "mutation_rate": mutation_rate,
        "popsize": popsize,
        "num_landscapes": num_strategy_landscapes,
        "num_reps": num_strategy_reps,
        "M": M,
        "strategy_grid_size": strategy_grid_size,
        "outer_reps": outer_reps,
        "thresholds": np.asarray(thresholds),
        "base_chances": np.asarray(base_chances),
        "splits": splits,
        "seed": seed,
    },
    "metadata": {
        "description": "N4A20 NK directed-evolution strategy sweep.",
        "paper_reference": "Figure 5A",
        "output_key": "nk_strategy_N4_A20",
        "filename": RAW_FILENAMES["nk_strategy_N4_A20"],
    },
}
save_raw(nk_strategy_N4_A20_payload, RAW_FILENAMES["nk_strategy_N4_A20"])

# Generation-count strategy sweeps
num_generation_landscapes = 10
for steps in tqdm(GENERATION_STEPS, desc="N4A20 generation-count strategy sweeps"):
    sweep = generate_nk_strategy_sweep(
        n_sites=N,
        num_alleles=A,
        k_values=K_values,
        mutation_rate=mutation_rate,
        popsize=popsize,
        num_landscapes=num_generation_landscapes,
        num_reps=num_strategy_reps,
        num_steps=int(steps),
        strategy_grid_size=strategy_grid_size,
        outer_reps=outer_reps,
        seed=seed,
    )
    generation_sweep_payload = {
        "data": sweep,
        "params": {
            "N": N,
            "A": A,
            "K_values": K_values,
            "mutation_rate": mutation_rate,
            "popsize": popsize,
            "num_landscapes": num_generation_landscapes,
            "num_reps": num_strategy_reps,
            "M": int(steps),
            "strategy_grid_size": strategy_grid_size,
            "outer_reps": outer_reps,
            "thresholds": np.asarray(thresholds),
            "base_chances": np.asarray(base_chances),
            "splits": splits,
            "seed": seed,
        },
        "metadata": {
            "description": "N4A20 generation-count directed-evolution strategy sweep.",
            "paper_reference": "Figure 5 support",
            "output_key": f"nk_strategy_N4_A20_steps{steps}",
            "filename": RAW_FILENAMES[f"nk_strategy_N4_A20_steps{steps}"],
        },
    }
    save_raw(generation_sweep_payload, RAW_FILENAMES[f"nk_strategy_N4_A20_steps{steps}"])


## Empirical Decay Curves - Figure 4

Generate uniform-start and all-start no-selection diffusion curves for GB1, TrpB, TEV, and ParD3.

- Paper reference: Figure 4 and empirical strategy-selection processing.
- Products: `empirical_decay_{name}_uniform` and `empirical_decay_{name}_all` for each landscape.
- Core variables: landscape name, total mutation rate, per-site mutation rate, population size, start policy, number of starts, replicates, and generations `M`.


In [ ]:
# Shared empirical decay parameters
empirical_landscapes = {name: load_empirical_landscape(name) for name in EMPIRICAL_NAMES}
mutation_rate = 0.1
num_reps = 10
M = 25
seed = 42

for name, landscape in empirical_landscapes.items():
    popsize = 60 if name == "ParD3" else 2500
    starts_count = 8000 if name == "ParD3" else 10000
    per_site_mutation_rate = mutation_rate / landscape.ndim

    uniform_starts = uniform_start_locs(landscape, num_starts=starts_count, seed=seed)
    uniform_decay = generate_empirical_decay_curves(
        landscape,
        mutation_rate=per_site_mutation_rate,
        popsize=popsize,
        starts=uniform_starts,
        num_reps=num_reps,
        num_steps=M,
        seed=seed,
    )
    uniform_decay_payload = {
        "data": uniform_decay,
        "params": {
            "name": name,
            "mutation_rate": mutation_rate,
            "per_site_mutation_rate": per_site_mutation_rate,
            "popsize": popsize,
            "starts_count": starts_count,
            "start_policy": "uniform",
            "num_reps": num_reps,
            "M": M,
            "seed": seed,
        },
        "metadata": {
            "description": f"{name} uniform-start empirical no-selection decay curves.",
            "paper_reference": "Figure 4",
            "output_key": f"empirical_decay_{name}_uniform",
            "filename": RAW_FILENAMES[f"empirical_decay_{name}_uniform"],
        },
    }
    save_raw(uniform_decay_payload, RAW_FILENAMES[f"empirical_decay_{name}_uniform"])

    all_starts = all_start_locs(landscape)
    all_decay = generate_empirical_decay_curves(
        landscape,
        mutation_rate=per_site_mutation_rate,
        popsize=popsize,
        starts=all_starts,
        num_reps=num_reps,
        num_steps=M,
        seed=seed,
    )
    all_decay_payload = {
        "data": all_decay,
        "params": {
            "name": name,
            "mutation_rate": mutation_rate,
            "per_site_mutation_rate": per_site_mutation_rate,
            "popsize": popsize,
            "starts_count": int(landscape.size),
            "start_policy": "all",
            "num_reps": num_reps,
            "M": M,
            "seed": seed,
        },
        "metadata": {
            "description": f"{name} all-start empirical no-selection decay curves.",
            "paper_reference": "Figure 4",
            "output_key": f"empirical_decay_{name}_all",
            "filename": RAW_FILENAMES[f"empirical_decay_{name}_all"],
        },
    }
    save_raw(all_decay_payload, RAW_FILENAMES[f"empirical_decay_{name}_all"])


## Empirical Strategy Sweeps - Figure 5D-G

Generate strategy sweeps from 100 uniformly sampled starts on each empirical landscape.

- Paper reference: Figure 5D-G.
- Product: `empirical_strategy_{name}_uniform` for each landscape.
- Core variables: landscape name, total population size, mutation rate, number of starts, strategy grid size, replicates, and generations `M`.


In [ ]:
# Shared empirical strategy parameters
num_starts = 100
mutation_rate = 0.025
popsize = 1200
num_reps = 10
M = 25
strategy_grid_size = 7
outer_reps = 10
seed = 42
thresholds, base_chances, splits = strategy_grid(strategy_grid_size)

for name, landscape in empirical_landscapes.items():
    starts = uniform_start_locs(landscape, num_starts=num_starts, seed=seed)
    strategy_sweep = generate_empirical_strategy_sweep(
        landscape,
        starts,
        mutation_rate=mutation_rate,
        popsize=popsize,
        num_reps=num_reps,
        num_steps=M,
        strategy_grid_size=strategy_grid_size,
        outer_reps=outer_reps,
        seed=seed,
    )
    strategy_sweep_payload = {
        "data": strategy_sweep,
        "params": {
            "name": name,
            "mutation_rate": mutation_rate,
            "popsize": popsize,
            "num_starts": num_starts,
            "start_policy": "uniform",
            "num_reps": num_reps,
            "M": M,
            "strategy_grid_size": strategy_grid_size,
            "outer_reps": outer_reps,
            "thresholds": np.asarray(thresholds),
            "base_chances": np.asarray(base_chances),
            "splits": splits,
            "seed": seed,
        },
        "metadata": {
            "description": f"{name} empirical directed-evolution strategy sweep from uniform starts.",
            "paper_reference": "Figure 5D-G",
            "output_key": f"empirical_strategy_{name}_uniform",
            "filename": RAW_FILENAMES[f"empirical_strategy_{name}_uniform"],
        },
    }
    save_raw(strategy_sweep_payload, RAW_FILENAMES[f"empirical_strategy_{name}_uniform"])


## Empirical Popsize Decay - Figure 4E

Generate empirical population-size sweeps for fitted squared-decay ruggedness estimates.

- Paper reference: Figure 4E.
- Product: `empirical_decay_{name}_popsize` for each landscape.
- Core variables: landscape name, total mutation rate, swept population sizes, effective population sizes, start count, replicates, and generations `M`.


In [ ]:
# Shared empirical population-size decay parameters
mutation_rate = 0.1
pop_sizes = np.linspace(25, 2500, 10, dtype=int)
num_reps = 10
M = 25
seed = 42

for name, landscape in empirical_landscapes.items():
    starts_count = 8000 if name == "ParD3" else 10000
    starts = uniform_start_locs(landscape, num_starts=starts_count, seed=seed)
    effective_pop_sizes = [max(1, int(popsize / 20)) if name == "ParD3" else int(popsize) for popsize in pop_sizes]
    popsize_results = []

    for popsize, effective_popsize in tqdm(zip(pop_sizes, effective_pop_sizes), total=len(pop_sizes), desc=f"{name} popsize decay"):
        popsize_results.append(
            generate_empirical_decay_curves(
                landscape,
                mutation_rate=mutation_rate / landscape.ndim,
                popsize=effective_popsize,
                starts=starts,
                num_reps=num_reps,
                num_steps=M,
                seed=seed,
            )
        )

    popsize_decay_payload = {
        "data": np.asarray(popsize_results),
        "params": {
            "name": name,
            "mutation_rate": mutation_rate,
            "per_site_mutation_rate": mutation_rate / landscape.ndim,
            "pop_sizes": pop_sizes,
            "effective_pop_sizes": effective_pop_sizes,
            "starts_count": starts_count,
            "start_policy": "uniform",
            "num_reps": num_reps,
            "M": M,
            "seed": seed,
        },
        "metadata": {
            "description": f"{name} empirical population-size no-selection decay sweep.",
            "paper_reference": "Figure 4E",
            "output_key": f"empirical_decay_{name}_popsize",
            "filename": RAW_FILENAMES[f"empirical_decay_{name}_popsize"],
        },
    }
    save_raw(popsize_decay_payload, RAW_FILENAMES[f"empirical_decay_{name}_popsize"])


## NK Heterogeneity - Figure 4

Generate the `N=4`, `A=20` NK heterogeneity comparison product used alongside empirical heterogeneity analyses.

- Paper reference: Figure 4B-F.
- Product: `nk_heterogeneity`, saved to `RAW_FILENAMES["nk_heterogeneity"]`.
- Core variables: `N`, `A`, `K_values`, mutation rate, population size, sampled starts, replicates, and generations `M`.


In [ ]:
# NK heterogeneity parameters
N, A = 4, 20
K_values = [1, 2, 3, 4]
mutation_rate = 0.1
popsize = 1200
num_starts = 10000
num_reps = 10
M = 25
seed = 42

nk_heterogeneity = generate_nk_decay_curves(
    n_sites=N,
    num_alleles=A,
    k_values=K_values,
    mutation_rate=mutation_rate,
    popsize=popsize,
    num_starts=num_starts,
    num_reps=num_reps,
    num_steps=M,
    seed=seed,
)
nk_heterogeneity_payload = {
    "data": nk_heterogeneity,
    "params": {
        "N": N,
        "A": A,
        "K_values": K_values,
        "mutation_rate": mutation_rate,
        "popsize": popsize,
        "num_starts": num_starts,
        "num_reps": num_reps,
        "M": M,
        "seed": seed,
    },
    "metadata": {
        "description": "N4A20 NK heterogeneity comparison decay curves.",
        "paper_reference": "Figure 4B-F",
        "output_key": "nk_heterogeneity",
        "filename": RAW_FILENAMES["nk_heterogeneity"],
    },
}
save_raw(nk_heterogeneity_payload, RAW_FILENAMES["nk_heterogeneity"])


## Missing Raw Products

After running the generation cells above, this reports any expected raw product that is still absent from `raw_data/`.

In [ ]:
missing = missing_raw_outputs()
if missing:
    print(f"Missing raw products: {len(missing)}")
    for key, path in missing.items():
        print(f"{key}: {path}")
else:
    print("All expected raw products are present.")

print(f"Expected raw products: {len(expected_raw_outputs())}")